# 深度學習框架應用（大學部）｜主題 10：循環神經網路：RNN、LSTM 與 GRU

這本筆記本配合投影片《主題 10：循環神經網路：RNN、LSTM 與 GRU》。五題由淺到深：先手算 RNN 的記憶、看懂三種循環層的形狀與參數量，再拿一條自己產生的時間序列比較基準線、線性模型和 RNN，接著親眼看梯度怎麼隨時間步消失、LSTM 怎麼留住它，最後訓練一個會寫字的字元級小模型。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 手算三步的 h，和 nn.RNN 對答案 | 第 125 頁 | 第 15～21 頁 |
| 二 | 三種循環層的形狀與參數量 | 第 126 頁 | 第 47～50、73～77 頁 |
| 三 | 基準線、線性模型、RNN 比一比 | 第 127 頁 | 第 28～45 頁 |
| 四 | 最後一步還感受得到第 1 步嗎 | 第 128 頁 | 第 52～56、66～67 頁 |
| 五 | 字元級語言模型，換三種溫度寫字 | 第 129 頁 | 第 99～106 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W10_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 由上往下跑：先跑「環境準備」，之後每一題如果有「準備（直接執行）」那一格，也要先跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 有 GPU 會自動用（「執行階段 › 變更執行階段類型」），沒有也沒關係：每一題在 CPU 上幾秒到兩分鐘就跑完。在 GPU 上跑，練習三、五的數字最後幾位會和投影片不一樣，大小與趨勢相同就對了。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

下面這一格五題都會用到：載入套件、決定用 GPU 還是 CPU，再加一個畫圖的小工具 `show`。

In [ ]:
# ===== 環境準備（直接執行）=====
import copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch：', torch.__version__, '｜裝置：', dev)

def show(curves, ylabel, xlabel='step', logy=False):
    """把好幾條曲線畫在同一張圖（圖上的字用英文，Colab 才不會變方塊）"""
    for name, ys in curves.items():
        plt.plot(range(1, len(ys) + 1), ys, 'o-', ms=3, label=name)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    if logy: plt.yscale('log')
    plt.grid(alpha=.3); plt.legend(); plt.show()

## 練習一：手算三步的 h，和 nn.RNN 對答案 (20%)

> 題目在投影片**第 125 頁**｜觀念在第 15～21 頁

**情境**：小智讀一段只有 3 步的序列：1、0.5、0。牠隨身帶一張小紙條 h，一開始是 0；每讀一步就用「新輸入×1.0＋舊紙條×0.5」，再過 tanh 壓到 −1～1 之間，改寫紙條。

**這題在練**：親手把 RNN 沿時間展開算一次，確認 nn.RNN 做的就是同一件事（權重每一步都共用）

**步驟**

1. 先執行「練習一：準備」：x 的形狀是 (1, 3, 1)，也就是 1 筆、3 步、每步 1 個數
2. 在 ✏️ 格填好激活函數，自己用迴圈算出 h_1、h_2、h_3
3. 建一個沒有偏差的 nn.RNN，把兩個權重設成一樣，看它算出來的三步和你的一不一樣

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一：準備（直接執行，不用改）=====
x = torch.tensor([[[1.0], [0.5], [0.0]]])  # 1 筆、3 個時間步、每步 1 個數
w_x, w_h = 1.0, 0.5                        # 新輸入乘 1.0、舊記憶乘 0.5
print('x 的形狀 (B, T, 特徵)：', tuple(x.shape))
print('三步的輸入：', x[0, :, 0].tolist())

def r4(xs):
    """把一串數字（或張量）攤平、四捨五入到小數第 4 位，印出來比較好讀"""
    if isinstance(xs, list): xs = torch.cat([v.reshape(-1) for v in xs])
    return [round(v, 4) for v in xs.detach().reshape(-1).tolist()]

In [ ]:
# ✏️ 練習一：自己算三步的 h，再請 nn.RNN 算一次
h, mine = torch.zeros(1), []     # h_0 = 0
for t in range(3):
    h = torch.___(w_x * x[0, t] + w_h * h)  # ✏️ 激活函數
    mine.append(h)               # 記下每一步的 h
print('自己算：', r4(mine))
rnn = nn.RNN(1, 1, bias=False, batch_first=___)  # ✏️
with torch.no_grad():
    rnn.weight_ih_l0.fill_(w_x)  # 新輸入的權重
    rnn.weight_hh_l0.fill_(___)  # ✏️ 舊記憶的權重
out, h_n = rnn(x)                # out: (1, 3, 1)
print('nn.RNN：', r4(out))
print('最後一步 = h_n？', r4(out[:, -1]) == r4(h_n))

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「自己算」「nn.RNN」兩行三個數字，以及「最後一步 = h_n？ True」

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 寫出 h_2 的算式（代入數字）。三步用的都是同一組 w_x、w_h，這叫什麼？好處是什麼？**

（在這裡寫）

**2. 第 3 步的輸入是 0，為什麼 h_3 不是 0？這說明 h 在做什麼？**

（在這裡寫）

## 練習二：三種循環層的形狀與參數量 (20%)

> 題目在投影片**第 126 頁**｜觀念在第 47～50、73～77 頁

**情境**：同一批資料 (8, 20, 3)：8 段序列、每段 20 個時間步、每步 3 個特徵。分別交給兩層、hidden＝16 的 nn.RNN、nn.GRU、nn.LSTM，再故意忘一次 batch_first、再試一次雙向。

**這題在練**：看懂循環層的輸入輸出形狀、h_n 的每一維是什麼，以及參數量為什麼是 1：3：4

**步驟**

1. 先執行「練習二：準備」，確認 x 的形狀是 (8, 20, 3)
2. 在 ✏️ 格填好第三種循環層、batch_first，以及數參數個數的方法
3. 對照輸出：out、h_n 各是幾維？忘了 batch_first 哪一個形狀變了？雙向又變成什麼？

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改）=====
torch.manual_seed(0)
x = torch.randn(8, 20, 3)     # 一批 8 段、每段 20 步、每步 3 個特徵
print('x 的形狀 (B, T, 特徵)：', tuple(x.shape))

In [ ]:
# ✏️ 練習二：三種循環層的形狀與參數量
for Cell in [nn.RNN, nn.GRU, nn.___]:   # ✏️ 第三種
    m = Cell(3, 16, 2, batch_first=___)  # ✏️ 2 層
    out, h = m(x)
    if isinstance(h, tuple): h = h[0]   # LSTM 多一個 c
    n = sum(p.___() for p in m.parameters())  # ✏️ 個數
    print(f'{Cell.__name__:4} out {tuple(out.shape)} '
          f'h_n {tuple(h.shape)} 參數 {n:,}')
out, (h, c) = nn.LSTM(3, 16, 2)(x)      # 忘了 batch_first
print('忘了 batch_first：h_n', tuple(h.shape))
out, (h, c) = nn.LSTM(3, 16, 2, batch_first=True,
                      bidirectional=True)(x)  # 雙向
print('雙向 out', tuple(out.shape), 'h_n', tuple(h.shape))

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行 out、h_n、參數量，以及「忘了 batch_first」「雙向」兩行形狀

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. nn.RNN 的 880 個參數怎麼算出來？為什麼 GRU、LSTM 剛好是它的 3 倍、4 倍？**

（在這裡寫）

**2. 忘了 batch_first，h_n 的形狀為什麼從 (2, 8, 16) 變成 (2, 20, 16)？雙向時要怎麼取最後的狀態？**

（在這裡寫）

## 練習三：基準線、線性模型、RNN 比一比 (20%)

> 題目在投影片**第 127 頁**｜觀念在第 28～45 頁

**情境**：一條 1,000 點的時間序列：週期 50 步的波，加上一點雜訊（標準差 0.2）。照時間切：前 800 點訓練、後 200 點測試；每個題目是 30 步的視窗，答案是下一點。

**這題在練**：先算「複製前一個值」的基準線，再訓練線性模型和簡單 RNN，用測試 RMSE 判斷模型有沒有真的學到東西

**步驟**

1. 先執行「練習三：準備」：會切出 770 個訓練視窗、200 個測試視窗，也準備好 `rmse` 和訓練用的 `fit`
2. 在 ✏️ 格填好基準線取哪一個點、簡單 RNN 的類別名稱、輸出層取哪一步
3. 比一比三個測試 RMSE，再看圖上前 100 點的預測曲線

> 這一題要訓練兩個模型，CPU 上大約 10～30 秒。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
rng = np.random.default_rng(0)                 # 固定亂數，大家的資料一樣
t = np.arange(1000)
s = np.sin(2 * np.pi * t / 50) + 0.2 * rng.standard_normal(1000)
s = torch.tensor(s, dtype=torch.float32)       # 週期 50 步的波＋一點雜訊
WIN, SPLIT = 30, 800                           # 視窗 30 步；前 800 點拿來練

def windows(a):
    """切成所有長度 WIN 的視窗 (N, WIN, 1)，答案是緊接在後的那一點 (N,)"""
    return a.unfold(0, WIN, 1)[:-1].unsqueeze(-1), a[WIN:]

Xtr, ytr = windows(s[:SPLIT])                  # 照時間切：前 800 點
Xte, yte = windows(s[SPLIT - WIN:])            # 後 200 點（往前多帶 30 點）
print('訓練視窗', tuple(Xtr.shape), '測試視窗', tuple(Xte.shape))

def rmse(p, y):
    return ((p - y) ** 2).mean().sqrt().item()

def fit(make, epochs=30, lr=3e-3):
    """固定種子建模型、練 30 回合，回傳測試段的預測（形狀和 yte 一樣）"""
    torch.manual_seed(0)
    m = make().to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True,
                    generator=torch.Generator().manual_seed(0))
    for ep in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            loss = F.mse_loss(m(xb).reshape(-1), yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    m.eval()
    with torch.no_grad():
        return m(Xte.to(dev)).reshape(-1).cpu()

def linear():
    """線性模型：30 步攤平成一排，乘一組權重再加起來"""
    return nn.Sequential(nn.Flatten(), nn.Linear(WIN, 1))

def report3(res, n=100):
    """印出每一種預測的測試 RMSE，再畫測試段前 n 點"""
    for k, p in res.items():
        print(f'{k:6} 測試 RMSE {rmse(p, yte):.3f}')
    plt.figure(figsize=(10, 3))
    plt.plot(yte[:n], 'k-', lw=2, label='actual')
    for k, p in res.items():
        plt.plot(p[:n], '--', label=k)
    plt.xlabel('test step'); plt.grid(alpha=.3); plt.legend(); plt.show()

In [ ]:
# ✏️ 練習三：基準線、線性模型、RNN 比一比
naive = Xte[:, ___, 0]          # ✏️ 視窗最後一個點
class RNNReg(nn.Module):
    def __init__(self, hid=32):
        super().__init__()
        self.rnn = nn.___(1, hid, batch_first=True)  # ✏️
        self.fc = nn.Linear(hid, 1)
    def forward(self, x):       # x: (B, 30, 1)
        out, h_n = self.rnn(x)  # (B, 30, 32)
        return self.fc(out[:, ___]).squeeze(-1)  # ✏️
res = {'naive': naive, 'linear': fit(linear),
       'RNN': fit(RNNReg)}      # 各練 30 回合
report3(res)                    # 印 RMSE、畫圖

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行測試 RMSE（naive、linear、RNN）和預測曲線圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 基準線的測試 RMSE 是多少？線性模型和 RNN 有沒有贏過它？差多少？**

（在這裡寫）

**2. 這條序列上，線性模型和 RNN 差不多好，為什麼？什麼情況 RNN 才會明顯比較好？**

（在這裡寫）

## 練習四：最後一步還感受得到第 1 步嗎？ (20%)

> 題目在投影片**第 128 頁**｜觀念在第 52～56、66～67 頁

**情境**：一段 50 步的序列交給 hidden＝32 的循環層，只看最後一步的輸出；量一量它對第 1～50 步的輸入各有多少梯度。梯度越小，代表越早的輸入對最後的答案越「沒感覺」。

**這題在練**：親眼看到梯度隨時間步消失，以及 LSTM 的遺忘閘開大時，梯度可以一路傳回第 1 步

**步驟**

1. 先執行「練習四：準備」：`grad_by_step(layer)` 會回傳 50 個數，第 1 步排最前面
2. 在 ✏️ 格建一個 LSTM，再複製一份，把遺忘閘 f 那一段偏差設成 3（σ(3)≈0.95）
3. 比一比三條曲線：RNN、預設的 LSTM、f≈0.95 的 LSTM，第 1 步各剩多少

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
def grad_by_step(layer, T=50):
    """最後一步的輸出，對第 1～T 步的輸入各有多少梯度（平均大小）"""
    g = torch.Generator().manual_seed(1)
    x = torch.randn(32, T, 1, generator=g, requires_grad=True)
    out, _ = layer(x)                # out: (32, T, hidden)
    out[:, -1].sum().backward()      # 只看最後一步
    return x.grad.abs().mean(dim=(0, 2)).tolist()   # 第 1 步排最前面

In [ ]:
# ✏️ 練習四：最後一步，還感受得到第 1 步嗎？
H = 32                                  # 隱藏狀態 32 維
torch.manual_seed(0)
rnn = nn.RNN(1, H, batch_first=True)
lstm = nn.___(1, H, batch_first=True)   # ✏️ 有閘門的
lstm95 = copy.deepcopy(lstm)            # 複製一份來改
with torch.no_grad():  # 偏差順序 i, f, g, o
    lstm95.bias_ih_l0[___:2 * H] = 3.0  # ✏️ f 那一段
g = {'RNN': grad_by_step(rnn), 'LSTM': grad_by_step(lstm),
     'LSTM f=0.95': grad_by_step(lstm95)}
for k, v in g.items():
    print(f'{k:11} 第50步 {v[-1]:.1e} 第40步 {v[-11]:.1e}'
          f' 第1步 {v[___]:.1e}')       # ✏️ 第 1 步
show(g, 'grad size', xlabel='time step', logy=True)

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行「第50步、第40步、第1步」的梯度，以及對數座標的三條曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. RNN 第 1 步的梯度比第 50 步小了幾個數量級？這和投影片哪一個概念有關？**

（在這裡寫）

**2. 預設的 LSTM 也一樣消失，為什麼把遺忘閘偏差設成 3 之後，梯度幾乎不掉？**

（在這裡寫）

## 練習五：字元級語言模型，換三種溫度寫字 (20%)

> 題目在投影片**第 129 頁**｜觀念在第 99～106 頁

**情境**：小智讀了莎士比亞的劇本（tinyshakespeare 的前 20 萬字，全部轉小寫），練習「猜下一個字元」。題目是一段 64 個字元，答案是同一段往後挪一格。練好以後，給它開頭「to be or not to b」，讓它一個字一個字往下寫。

**這題在練**：看懂字元級語言模型的資料與輸出，並用溫度控制寫出來的字有多大膽

**步驟**

1. 先執行「練習五：準備＋訓練」：會下載約 1 MB 的文字、建字典、訓練 800 步，印出 loss
2. 在 ✏️ 格填好：取最後一步的分數、分數除以溫度、照機率抽一個字
3. 比一比溫度 0.2、1.0、2.0 寫出來的三段字

> 準備格要訓練 800 步，CPU 上大約 1～2 分鐘（有 GPU 更快）。在 GPU 上跑，寫出來的字會和投影片不一樣，「低溫重複、高溫亂碼」的樣子相同就對了。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備＋訓練（直接執行，不用改，約 1～2 分鐘）=====
import os, urllib.request
URL = ('https://raw.githubusercontent.com/karpathy/char-rnn/'
       'master/data/tinyshakespeare/input.txt')
if not os.path.exists('shakespeare.txt'):
    urllib.request.urlretrieve(URL, 'shakespeare.txt')
text = open('shakespeare.txt', encoding='utf-8').read().lower()  # 全部轉小寫
vocab = sorted(set(text))                      # 不重複的字元，排好
stoi = {c: i for i, c in enumerate(vocab)}     # 字元 → 編號

def encode(s):
    return torch.tensor([stoi[c] for c in s.lower()], device=dev)

data = encode(text[:200_000])                  # 前 20 萬字拿來練
print('全文', len(text), '字；字典', len(vocab), '個字元：', repr(''.join(vocab)))
w = text[:12]
print('題目', repr(w[:-1]), '→ 答案', repr(w[1:]))   # 答案往後挪一格

class CharModel(nn.Module):
    """Embedding → GRU → Linear：每一步都猜下一個字元"""
    def __init__(self, V, emb=16, hid=128):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.gru = nn.GRU(emb, hid, batch_first=True)
        self.fc = nn.Linear(hid, V)
    def forward(self, x):                      # x: (B, T) 的整數
        out, _ = self.gru(self.emb(x))         # (B, T, 128)
        return self.fc(out)                    # (B, T, 39)：每一步 39 個分數

def train_char(model, steps=800, WIN=64, bs=64):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    g = torch.Generator().manual_seed(0)
    for step in range(1, steps + 1):
        i = torch.randint(0, len(data) - WIN - 1, (bs,), generator=g).tolist()
        x = torch.stack([data[k:k + WIN] for k in i])           # 題目
        y = torch.stack([data[k + 1:k + WIN + 1] for k in i])   # 答案：往後挪一格
        loss = F.cross_entropy(model(x).reshape(-1, len(vocab)), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if step in (1, 200, 400, 600, 800):
            print(f'第 {step:3} 步  loss {loss.item():.3f}')

torch.manual_seed(0)
model = CharModel(len(vocab)).to(dev)
train_char(model)
model.eval()

In [ ]:
# ✏️ 練習五：讓模型寫字，換三種溫度
@torch.no_grad()                          # 寫字不用算梯度
def next_char(s, temp):
    logits = model(encode(s)[None])[0, ___]  # ✏️ 最後一步
    p = F.softmax(logits / ___, dim=-1)   # ✏️ 除以溫度
    return vocab[torch.___(p, 1)]         # ✏️ 照機率抽
start = 'to be or not to b'
with torch.no_grad():
    best = model(encode(start)[None])[0, -1].argmax()
print('最可能的下一個字：', repr(vocab[best]))
for temp in [0.2, 1.0, 2.0]:
    torch.manual_seed(0); s = start       # 大家抽到的一樣
    for _ in range(80): s += next_char(s, temp)
    print(f'--- 溫度 {temp} ---'); print(s)

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：準備格印出的 loss（往下降）、「最可能的下一個字」，以及三種溫度寫出來的三段字

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 第 1 步的 loss 大約是 3.66，剛好是 ln 39，為什麼？練完以後降到多少？**

（在這裡寫）

**2. 溫度 0.2、1.0、2.0 寫出來的字各有什麼特色？為什麼會這樣？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：LSTM 的遺忘閘是什麼） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W10_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。